### PASO 1:
Importar librerias necesarias

In [115]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LinearRegression
from colorama import Fore, init

init(autoreset=True, strip=False, convert=False)

### PASO 2:
Cargar dataset de Redes Sociales

In [116]:
# #TODO: Carga los conjuntos de datos correspondientes:
data_set = pd.read_csv('redes_sociales.csv')

### PASO3:
Identificar qué columnas contienen valores faltantes y su porcentaje

In [117]:
# #TODO: Columnas que contienen valores faltantes:
print(Fore.YELLOW +"\nValores NULL por columna:")
print(data_set.isnull().sum())


Valores NULL por columna:
edad                    12
red_social_principal     9
frecuencia_uso          10
horas_diarias            6
sesiones_diarias         6
tipo_contenido           8
interaccion_diaria      10
publica_contenido       10
uso_nocturno            11
notificaciones          13
motivo_principal        12
nivel_satisfaccion      11
dtype: int64


In [118]:
# # TODO: Identificar y mostrar el porcentaje de datos faltantes:
#* porcentaje = (datos_faltantes / numero._filas) * 100
missing_percentage1 = round( (data_set.isnull().sum() / data_set.shape[0]) * 100, 2 )

#* Imprimir los porcentajes:
print(Fore.YELLOW + "Porcentaje de datos faltantes en dataset:")
print(missing_percentage1.to_string())

Porcentaje de datos faltantes en dataset:
edad                    12.0
red_social_principal     9.0
frecuencia_uso          10.0
horas_diarias            6.0
sesiones_diarias         6.0
tipo_contenido           8.0
interaccion_diaria      10.0
publica_contenido       10.0
uso_nocturno            11.0
notificaciones          13.0
motivo_principal        12.0
nivel_satisfaccion      11.0


### PASO 3:
Separar por categorias numericas y de Texto

In [119]:
cat_col = data_set.select_dtypes(include=['object', 'str']).columns
num_col = data_set.select_dtypes(include=np.number).columns

print(Fore.CYAN + 'Categorical columns:\n', cat_col)
print(Fore.CYAN + 'Numerical columns:\n', num_col)

Categorical columns:
 Index(['red_social_principal', 'frecuencia_uso', 'tipo_contenido',
       'publica_contenido', 'uso_nocturno', 'notificaciones',
       'motivo_principal', 'nivel_satisfaccion'],
      dtype='str')
Numerical columns:
 Index(['edad', 'horas_diarias', 'sesiones_diarias', 'interaccion_diaria'], dtype='str')


### PASO 4:
Remplazar los datos faltantes del dataset:
- Categoricos por moda
- Numericos por mediana

Volver a comprobar los valores faltantes estén en 0%

In [120]:
# #TODO: Iterar categorias y remplazar los faltantes por la mode de las columnas
for col in cat_col:
    mode = data_set[col].mode()[0]
    data_set[col] = data_set[col].fillna(mode)

#TODO: Iterar categorias numericas y remplazar los faltantes por la media de las columnas
for col in num_col:
    median = data_set[col].median()
    data_set[col] = data_set[col].fillna(median)


print(Fore.YELLOW + "Porcentaje de datos faltantes en dataset:")
print( (round((data_set.isnull().sum() / data_set.shape[0]) * 100, 2) ).to_string() )

Porcentaje de datos faltantes en dataset:
edad                    0.0
red_social_principal    0.0
frecuencia_uso          0.0
horas_diarias           0.0
sesiones_diarias        0.0
tipo_contenido          0.0
interaccion_diaria      0.0
publica_contenido       0.0
uso_nocturno            0.0
notificaciones          0.0
motivo_principal        0.0
nivel_satisfaccion      0.0


### PASO 5:
Define las variables:
- Variables de entrada -> X
- Variable de salida -> Y

In [122]:
# #* Guarda una copia del DataFrame eliminando la columna 'publica_contenido'
x = data_set.drop('publica_contenido', axis=1)
#* Guarda solo la columna 'publica_contenido' y mapear respuestas en 0 y 1
y = data_set['publica_contenido'].map({
    'No': 0,
    'Sí': 1
})

#* Actualizar la Separar de columnas
cat_col = x.select_dtypes(include=['object', 'str']).columns
num_col = x.select_dtypes(include=np.number).columns

print(Fore.CYAN + "Categóricas:", cat_col)
print(Fore.CYAN + "Numéricas:", num_col)

Categóricas: Index(['red_social_principal', 'frecuencia_uso', 'tipo_contenido',
       'uso_nocturno', 'notificaciones', 'motivo_principal',
       'nivel_satisfaccion'],
      dtype='str')
Numéricas: Index(['edad', 'horas_diarias', 'sesiones_diarias', 'interaccion_diaria'], dtype='str')


### PASO 6:
Dividir el dataset en:
- 60 % entrenamiento
- 20 % validación
- 20 % prueba

In [124]:
# #TODO: Dividir en entrenamiento (60%) y temporal (40%)
x_train, x_temp, y_train, y_temp = train_test_split(
    x, y,
    test_size=0.40,
    random_state=4
)

#TODO: Dividir el 40% restante en validación (20%) y prueba (20%)
x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, 
    test_size=0.50, 
    random_state=42
)

#* Mostrar distribución de percentajes
print("Entrenamiento:", x_train.shape[0])
print("Validación:", x_val.shape[0])
print("Prueba:", x_test.shape[0])

Entrenamiento: 60
Validación: 20
Prueba: 20


### PASO 7:
Obtener:
- Matriz de confusión
- Porcentajes de Acuracy
- Precision
- Recall
- F1-Score

In [125]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_col),
        ('num', 'passthrough', num_col)
    ]
)
print(preprocessor)

ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                 Index(['red_social_principal', 'frecuencia_uso', 'tipo_contenido',
       'uso_nocturno', 'notificaciones', 'motivo_principal',
       'nivel_satisfaccion'],
      dtype='str')),
                                ('num', 'passthrough',
                                 Index(['edad', 'horas_diarias', 'sesiones_diarias', 'interaccion_diaria'], dtype='str'))])


In [126]:
x_train = preprocessor.fit_transform(x_train)

x_val = preprocessor.transform(x_val)

x_test = preprocessor.transform(x_test)

In [127]:
model = LinearRegression()

model.fit(x_train, y_train)

print(f"Intercept: {model.intercept_}")
print(f"Coefficient: {model.coef_}")

Intercept: 0.27831890516234475
Coefficient: [-1.14761182e-01  2.35149159e-01 -1.89181285e-01  5.10262171e-02
 -7.81391800e-02  9.59062705e-02  2.60299635e-01 -6.72375065e-01
  2.00380773e-01  1.11661798e-01  1.00032860e-01 -4.87294669e-01
  1.52389244e-01  2.67500038e-01 -4.49789268e-02  1.35393193e-01
 -1.27891619e-01  2.59014170e-01 -1.54131431e-01  4.18387523e-02
 -9.79183317e-02  5.28560306e-01 -5.28541084e-01  5.60603574e-02
  1.05101201e-01 -2.83424924e-01  3.43633178e-01 -1.65309455e-01
 -1.26513567e-02  8.18620254e-02 -1.33477978e-01  9.13364756e-02
 -1.40257366e-01  1.13188201e-01  2.01313583e-01 -5.01978599e-01
 -7.08111770e-02  3.94142814e-01 -2.26666214e-02 -3.90167019e-03
 -4.01641725e-02  3.16686350e-02  5.20791404e-04]


In [128]:
y_pred = model.predict(x_test)
y_pred = (y_pred >= 0.5).astype(int)

In [129]:
# #* Matriz de confución:
matriz = confusion_matrix(y_test, y_pred)
#* Exactitud
accuracy = accuracy_score(y_test, y_pred)
#* Precisión
precision = precision_score(y_test, y_pred)
#* Sensibilidad
recall = recall_score(y_test, y_pred)
#* F1-Score
f1 = f1_score(y_test, y_pred)

In [130]:
print("Matriz de confusión:")
print(matriz)

print(f"\nAccuracy:  {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1-Score:  {f1 * 100:.2f}%")

Matriz de confusión:
[[9 5]
 [4 2]]

Accuracy:  55.00%
Precision: 28.57%
Recall:    33.33%
F1-Score:  30.77%
